Create a Dataset for @HOME2024 using GroundingDino and SAM

In [5]:
import os
import random
import json
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont
from pycocotools import mask
import json
import yaml
import csv
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import ultralytics
import time
import imutils
import argparse

#grounding imports----------------

import groundingdino.datasets.transforms as T
from groundingdino.models import build_model
from groundingdino.util import box_ops
from groundingdino.util.slconfig import SLConfig
from groundingdino.util.utils import clean_state_dict, get_phrases_from_posmap
from groundingdino.util.vl_utils import create_positive_map_from_span


ModuleNotFoundError: No module named 'pycocotools'

In [7]:
#resize images in a folder to a specific size

pathtofiles = "/home/jabv/Desktop/bolsas/"
pathtoimage = [f.path for f in os.scandir(pathtofiles) if f.is_dir()]
size = 720

#pathtoimage = os.listdir(pathtofiles)

for filepath in pathtoimage:
    folder = filepath + "/"
    for filename in os.listdir(folder):
        img = Image.open(folder + filename)
        img = img.resize((size, size))
        img.save(folder + filename)

Auto label con Segment anyting y modelo de YOLOv8

In [2]:
pathtofiles = "/home/jabv/Desktop/bolsas/" #path to images to process
resultspath = "/home/jabv/Desktop/bolsas_precut" #path to save results all ready processed and segmented images
if not os.path.exists(resultspath):
    os.makedirs(resultspath)

def load_image(image_path):

    image_pil = Image.open(image_path).convert("RGB")  # load image

    transform = T.Compose(
        [
            T.RandomResize([800], max_size=1333),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )
    image, _ = transform(image_pil, None)  # 3, h, w
    return image_pil, image


def load_model(model_config_path, model_checkpoint_path, cpu_only=False):
    args = SLConfig.fromfile(model_config_path)
    args.device = "cuda" if not cpu_only else "cpu"
    model = build_model(args)
    checkpoint = torch.load(model_checkpoint_path, map_location="cpu")
    load_res = model.load_state_dict(clean_state_dict(checkpoint["model"]), strict=False)
    print(load_res)
    _ = model.eval()
    return model


def get_grounding_output(model, image, caption, box_threshold, text_threshold=None, with_logits=True, cpu_only=False, token_spans=None):
    assert text_threshold is not None or token_spans is not None, "text_threshould and token_spans should not be None at the same time!"
    caption = caption.lower()
    caption = caption.strip()
    if not caption.endswith("."):
        caption = caption + "."
    device = "cuda" if not cpu_only else "cpu"
    model = model.to(device)
    image = image.to(device)
    with torch.no_grad():
        print("Running model...")
        outputs = model(image[None], captions=[caption])
    logits = outputs["pred_logits"].sigmoid()[0]  # (nq, 256)
    boxes = outputs["pred_boxes"][0]  # (nq, 4)

    # filter output
    if token_spans is None:
        logits_filt = logits.cpu().clone()
        boxes_filt = boxes.cpu().clone()
        filt_mask = logits_filt.max(dim=1)[0] > box_threshold
        logits_filt = logits_filt[filt_mask]  # num_filt, 256
        boxes_filt = boxes_filt[filt_mask]  # num_filt, 4

        # get phrase
        tokenlizer = model.tokenizer
        tokenized = tokenlizer(caption)
        # build pred
        pred_phrases = []
        for logit, box in zip(logits_filt, boxes_filt):
            pred_phrase = get_phrases_from_posmap(logit > text_threshold, tokenized, tokenlizer)
            if with_logits:
                pred_phrases.append(pred_phrase + f"({str(logit.max().item())[:4]})")
            else:
                pred_phrases.append(pred_phrase)
    else:
        # given-phrase mode
        positive_maps = create_positive_map_from_span(
            model.tokenizer(text_prompt),
            token_span=token_spans
        ).to(image.device) # n_phrase, 256

        logits_for_phrases = positive_maps @ logits.T # n_phrase, nq
        all_logits = []
        all_phrases = []
        all_boxes = []
        for (token_span, logit_phr) in zip(token_spans, logits_for_phrases):
            # get phrase
            phrase = ' '.join([caption[_s:_e] for (_s, _e) in token_span])
            # get mask
            filt_mask = logit_phr > box_threshold
            # filt box
            all_boxes.append(boxes[filt_mask])
            # filt logits
            all_logits.append(logit_phr[filt_mask])
            if with_logits:
                logit_phr_num = logit_phr[filt_mask]
                all_phrases.extend([phrase + f"({str(logit.item())[:4]})" for logit in logit_phr_num])
            else:
                all_phrases.extend([phrase for _ in range(len(filt_mask))])
        boxes_filt = torch.cat(all_boxes, dim=0).cpu()
        pred_phrases = all_phrases


    return boxes_filt, pred_phrases

# cfg
config_file = "/home/jabv/Desktop/home-vision/dataset_generator/groundingdino/config/GroundingDINO_SwinT_OGC.py"  # change the path of the model config file

#wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
checkpoint_path = "/home/jabv/Desktop/home-vision/dataset_generator/groundingdino_swint_ogc.pth"  # change the path of the model
text_prompt = "bannana"
output_dir = resultspath
box_threshold = 0.3
text_threshold = 0.25
token_spans = None



import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamPredictor
sam_model = "h"

#use sam model
#wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
#wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth
if sam_model =="h":
  sam_checkpoint = "sam_vit_h_4b8939.pth"
  model_type = "vit_h"
else:
  sam_checkpoint = "sam_vit_l_0b3195.pth"
  model_type = "vit_l"

device = "cuda"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

images=[]
annotations=[]
categories=[]

img_id=0
anno_id=0

#check if results directory exists, else create it
if not os.path.exists(resultspath):
  os.makedirs(resultspath)

#make a list of all the directories in the path
pathtoimage = [f.path for f in os.scandir(pathtofiles) if f.is_dir()]

#pathtoimage = os.listdir(pathtofiles)

for filepath in pathtoimage:
    imgPaths = os.listdir(filepath)
    print(imgPaths)

    i=0

    for imgPath in imgPaths:
        print(f"Processing image: {imgPath}")
        img = imutils.resize(cv2.imread(f"{filepath}/{imgPath}"))
        if img is None:
            continue

    #------------------------start grounding----------------------------------------------
        #image_path = args.image_path

        # load image
        image_pil, image = load_image(f"{filepath}/{imgPath}")

        # load model
        model = load_model(config_file, checkpoint_path, cpu_only=False)

        # set the text_threshold to None if token_spans is set.
        if token_spans is not None:
            text_threshold = None
            print("Using token_spans. Set the text_threshold to None.")

        # run model
        boxes_filt, pred_phrases = get_grounding_output(
            model, image, text_prompt, box_threshold, text_threshold, cpu_only=False, token_spans=eval(f"{token_spans}")
        )

        #found bb dimensions

        size = image_pil.size
        pred_dict = {
            "boxes": boxes_filt,
            "size": [size[1], size[0]],  # H,W
            "labels": pred_phrases,
        }

        H, W = pred_dict["size"]
        boxes = pred_dict["boxes"]
        labels = pred_dict["labels"]
        assert len(boxes) == len(labels), "boxes and labels must have same length"

        draw = ImageDraw.Draw(image_pil)
        mask = Image.new("L", image_pil.size, 0)
        mask_draw = ImageDraw.Draw(mask)

        #change pil image to cv2 image
        img = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
        img2 = img.copy()
        # draw boxes and masks
        for box, label in zip(boxes, labels):
            # from 0..1 to 0..W, 0..H
            box = box * torch.Tensor([W, H, W, H])
            # from xywh to xyxy
            box[:2] -= box[2:] / 2
            box[2:] += box[:2]
            # random color
            color = tuple(np.random.randint(0, 255, size=1).tolist())
            # draw
            padding = 10
            x0, y0, x1, y1 = box
            x0, y0, x1, y1 = int(x0)-padding, int(y0)-padding, int(x1)+padding, int(y1)+padding

            #validate if the bounding box is inside the image
            if x0 < 0:
                x0 = 0
            if y0 < 0:
                y0 = 0
            if x1 > W:
                x1 = W
            if y1 > H:
                y1 = H
                
            #draw rectangles
            cv2.rectangle(img2, (x0, y0), (x1, y1), color, 2)

            draw.rectangle([x0, y0, x1, y1], outline=color, width=6)
            # draw.text((x0, y0), str(label), fill=color)

            font = ImageFont.load_default()
            if hasattr(font, "getbbox"):
                bbox = draw.textbbox((x0, y0), str(label), font)
            else:
                w, h = draw.textsize(str(label), font)
                bbox = (x0, y0, w + x0, y0 + h)
            # bbox = draw.textbbox((x0, y0), str(label))
            draw.rectangle(bbox, fill=color)
            draw.text((x0, y0), str(label), fill="white")

            mask_draw.rectangle([x0, y0, x1, y1], fill=255, width=6)
        
    # ----------------End grounding ---------------------------------------------------------   
        
    # ----------------Start SAM--------------------------------------------------------------  
            
            class_name = filepath.split("/")[-1]
            #print x0, y0, x1, y1
            print(f"Bounding box: {x0}, {y0}, {x1}, {y1}")

            sam_bounding_box = np.array([x0, y0, x1, y1])
            ran_sam = False
            #run sam
            if ran_sam == False:
                predictor.set_image(img)
                ran_sam = True

            mask, _, _ = predictor.predict(
                point_coords=None,
                point_labels=None,
                box=sam_bounding_box,
                multimask_output=False,
            )

            mask, _, _ = predictor.predict(box=sam_bounding_box, multimask_output=False)

            #Make png mask
            contours, _ = cv2.findContours(mask[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) # Your call to find the contours

            # threshold input image using otsu thresholding as mask and refine with morphology
            ret, pngmask = cv2.threshold(mask[0].astype(np.uint8), 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU) 
            kernel = np.ones((9,9), np.uint8)
            pngmask = cv2.morphologyEx(pngmask, cv2.MORPH_CLOSE, kernel)
            pngmask = cv2.morphologyEx(pngmask, cv2.MORPH_OPEN, kernel)
            result = img.copy()
            result = cv2.cvtColor(result, cv2.COLOR_BGR2BGRA)
            result[:, :, 3] = pngmask                           

    # ----------------End SAM-----------------------------------------------------------------  
            #cv2.imwrite(f"{resultspath}/groundingcv2_{imgPath}", img2)

            #image_pil.save(f"{resultspath}/grounding_{imgPath}")

            if not os.path.exists(f"{resultspath}/{class_name}"):
                os.mkdir(f"{resultspath}/{class_name}")

            file_path = f"{resultspath}/{class_name}/{imgPath[:-4]}.png"
            if os.path.exists(file_path):
                if os.path.exists(f"{resultspath}/{class_name}/{imgPath[:-4]}_1.png"):
                    print("File already exists, saving with _2")
                    cv2.imwrite(f"{resultspath}/{class_name}/{imgPath[:-4]}_2.png", result)
                else:
                    print("File already exists, saving with _1")
                    file_path = f"{resultspath}/{class_name}/{imgPath[:-4]}_1.png"

            cv2.imwrite(file_path, result)
            i=i+1
            ran_sam = False

['IMG_4947.png', 'IMG_4936.png', 'IMG_4638.png', 'IMG_4563.png', 'IMG_4948.png', 'IMG_4926.png', 'IMG_4966.png', 'IMG_4610.png', 'IMG_4572.png', 'IMG_4953.png', 'IMG_4962.png', 'IMG_4559.png', 'IMG_4883.png', 'IMG_4637.png', 'IMG_4920.png', 'IMG_4579.png', 'IMG_4560.png', 'IMG_4557.png', 'IMG_4574.png', 'IMG_4886.png', 'IMG_4967.png', 'IMG_4631.png', 'IMG_4937.png', 'IMG_4545.png', 'IMG_4969.png', 'IMG_4944.png', 'IMG_4605.png', 'IMG_4873.png', 'IMG_4951.png', 'IMG_4613.png', 'IMG_4548.png', 'IMG_4974.png', 'IMG_4941.png', 'IMG_4589.png', 'IMG_4614.png', 'IMG_4931.png', 'IMG_4907.png', 'IMG_4914.png', 'IMG_4655.png', 'IMG_4906.png', 'IMG_4599.png', 'IMG_4970.png', 'IMG_4587.png', 'IMG_4542.png', 'IMG_4896.png', 'IMG_4956.png', 'IMG_4630.png', 'IMG_4583.png', 'IMG_4938.png', 'IMG_4913.png', 'IMG_4633.png', 'IMG_4977.png', 'IMG_4904.png', 'IMG_4573.png', 'IMG_4881.png', 'IMG_4540.png', 'IMG_4901.png', 'IMG_4624.png', 'IMG_4628.png', 'IMG_4909.png', 'IMG_4618.png', 'IMG_4983.png', 'IMG_45

/home/jabv/.local/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased
_IncompatibleKeys(missing_keys=[], unexpected_keys=['label_enc.weight', 'bert.embeddings.position_ids'])
Running model...


/home/jabv/.local/lib/python3.8/site-packages/transformers/modeling_utils.py:962: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/jabv/.local/lib/python3.8/site-packages/torch/utils/checkpoint.py:31: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn("None of the inputs have requires_grad=True. Gradients will be None")


Bounding box: 253, 1031, 2375, 2702
Bounding box: 244, 1024, 2781, 2702
File already exists, saving with _1
Bounding box: 1070, 1471, 1849, 2212
File already exists, saving with _2
Processing image: IMG_4936.png
final text_encoder_type: bert-base-uncased
_IncompatibleKeys(missing_keys=[], unexpected_keys=['label_enc.weight', 'bert.embeddings.position_ids'])
Running model...


...

libpng error: Write Error


Bounding box: 1410, 1692, 2155, 2370


libpng error: Write Error


Processing image: IMG_4586.png


OSError: [Errno 28] No space left on device: '/tmp/tmpq_d0hkqy'

In [37]:
from PIL import Image
import os

datasetpath = "/home/jabv/Desktop/prueba/res/"
resultspath = "/home/jabv/Desktop/prueba/DS_res/"

#create a result folder if it doesn't exist
if not os.path.exists(resultspath):
    os.makedirs(resultspath)
    os.makedirs(resultspath+"gatos/")
    os.makedirs(resultspath+"jarra/")
    os.makedirs(resultspath+"lata/")
    os.makedirs(resultspath+"manzana/")
    os.makedirs(resultspath+"platano/")
    os.makedirs(resultspath+"silla/")


fg_folders = [
    ("gatos/"),
    ("jarra/"),
    ("lata/"),
    ("manzana/"),
    ("platano/"),
    ("silla/")
]

for folder in fg_folders:
    for filename in os.listdir(f"{datasetpath}{folder}"):
        try:
            print(f"{filename} started")
            myImage = Image.open(datasetpath+folder+filename)
            black = Image.new('RGBA', myImage.size)
            myImage = Image.composite(myImage, black, myImage)
            #print("aqui")
            myCroppedImage = myImage.crop(myImage.getbbox())
            myCroppedImage.save(f"{resultspath}{folder}{filename}")
            print(f"{filename} done")
        except:
            print(f"{filename} failed")
            continue
print("all done")

gatos..png started
aqui
gatos..png done
gatos._2.png started
aqui
gatos._2.png done
gatos._1.png started
aqui
gatos._1.png done
jarra3..png started
aqui
jarra3..png done
jarra2..png started
aqui
jarra2..png done
jarra1._2.png started
aqui
jarra1._2.png done
jarra1._1.png started
aqui
jarra1._1.png done
jarra1..png started
aqui
jarra1..png done
lata.png started
aqui
lata.png done
manzana1..png started
aqui
manzana1..png done
manzana2._1.png started
aqui
manzana2._1.png done
manzana1._2.png started
aqui
manzana1._2.png done
manzana2._2.png started
aqui
manzana2._2.png done
manzana1._1.png started
aqui
manzana1._1.png done
manzana2..png started
aqui
manzana2..png done
platano3..png started
aqui
platano3..png done
platano3._1.png started
aqui
platano3._1.png done
platano1._1.png started
aqui
platano1._1.png done
platano2..png started
aqui
platano2..png done
platano2._1.png started
aqui
platano2._1.png done
platano1..png started
aqui
platano1..png done
silla.png started
aqui
silla.png done


In [1]:
import os
import cv2
import numpy as np
import random
import json
import yaml
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
import math
import kagglehub

/home/daniela/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define the paths to the three folders containing the images
datasetPath = "/home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/baseDataSet"  
fg_folders = [
    (f"{datasetPath}/pattern1/","pattern1" ),
    (f"{datasetPath}/pattern2/","pattern2" ),
    (f"{datasetPath}/pattern3/","pattern3" ),
    (f"{datasetPath}/pattern4/","pattern4" ),
    (f"{datasetPath}/pattern5/","pattern5" ),
    (f"{datasetPath}/pattern6/","pattern6" ),
    (f"{datasetPath}/pattern7/","pattern7" ),
    (f"{datasetPath}/pattern8/","pattern8" ),
    (f"{datasetPath}/pattern9/","pattern9" ),
    (f"{datasetPath}/pattern10/","pattern10" ),
    (f"{datasetPath}/pattern11/","pattern11" ),
    (f"{datasetPath}/pattern12/","pattern12" ),
    (f"{datasetPath}/pattern13/","pattern13" ),
    (f"{datasetPath}/pattern14/","pattern14" ),
    (f"{datasetPath}/pattern15/","pattern15" ),
    (f"{datasetPath}/pattern16/","pattern16" ),
    (f"{datasetPath}/pattern17/","pattern17" ),
    (f"{datasetPath}/pattern18/","pattern18" ),
    (f"{datasetPath}/pattern19/","pattern19" ),
    (f"{datasetPath}/pattern20/","pattern20" )
]
bg_folder = "/home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/bgImages/"
output_folder = "/home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/finalOutput/"

In [3]:
objects_list = ["pattern1", "pattern2", "pattern3", "pattern4", "pattern5", "pattern6", "pattern7", "pattern8", "pattern9", "pattern10", "pattern11", "pattern12", "pattern13", "pattern14", "pattern15","pattern16","pattern17", "pattern18", "pattern19", "pattern20"]
annotations_ID = {}
categories = []
for i, object in enumerate(objects_list):
    annotations_ID[object] = i
    categories.append({"id": i, "name": object})

print(annotations_ID)
print(categories)

{'pattern1': 0, 'pattern2': 1, 'pattern3': 2, 'pattern4': 3, 'pattern5': 4, 'pattern6': 5, 'pattern7': 6, 'pattern8': 7, 'pattern9': 8, 'pattern10': 9, 'pattern11': 10, 'pattern12': 11, 'pattern13': 12, 'pattern14': 13, 'pattern15': 14, 'pattern16': 15, 'pattern17': 16, 'pattern18': 17, 'pattern19': 18, 'pattern20': 19}
[{'id': 0, 'name': 'pattern1'}, {'id': 1, 'name': 'pattern2'}, {'id': 2, 'name': 'pattern3'}, {'id': 3, 'name': 'pattern4'}, {'id': 4, 'name': 'pattern5'}, {'id': 5, 'name': 'pattern6'}, {'id': 6, 'name': 'pattern7'}, {'id': 7, 'name': 'pattern8'}, {'id': 8, 'name': 'pattern9'}, {'id': 9, 'name': 'pattern10'}, {'id': 10, 'name': 'pattern11'}, {'id': 11, 'name': 'pattern12'}, {'id': 12, 'name': 'pattern13'}, {'id': 13, 'name': 'pattern14'}, {'id': 14, 'name': 'pattern15'}, {'id': 15, 'name': 'pattern16'}, {'id': 16, 'name': 'pattern17'}, {'id': 17, 'name': 'pattern18'}, {'id': 18, 'name': 'pattern19'}, {'id': 19, 'name': 'pattern20'}]


In [4]:
# Load the list of files in each of the three folders
fg_files = {}
for folder, category in fg_folders:
    fg_files[category] = os.listdir(folder)

#### Get background dataset

In [6]:
# Download latest version
dataset_path = kagglehub.dataset_download("balraj98/stanford-background-dataset")

os.system(f'cp "{dataset_path}/images/"* /home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/bgImages/')

0

In [8]:
# Ruta de la carpeta
folder_path = "/home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/bgImages/"

# Contar archivos
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])

print(f"Hay {file_count} archivos en la carpeta.")

Hay 153 archivos en la carpeta.


#### Check background images sizes

In [9]:
backgrounds = os.listdir(bg_folder)

for bg in backgrounds:
    pattth = os.path.join(bg_folder, bg)
    with Image.open(pattth) as imagen:
        print(f"{bg} {imagen.size}")  # Output will be (width, height)

6000344.jpg (320, 240)
9003585.jpg (214, 320)
9004520.jpg (320, 240)
6000330.jpg (320, 240)
9003301.jpg (320, 240)
6000337.jpg (320, 240)
9001001.jpg (320, 240)
6000316.jpg (320, 240)
6000332.jpg (320, 240)
6000355.jpg (320, 240)
6000313.jpg (320, 240)
9001071.jpg (320, 213)
6000298.jpg (320, 240)
football-pitch.png (640, 480)
6000324.jpg (320, 240)
9004383.jpg (320, 240)
DifCanchaVSSS.png (640, 480)
canchaVSSS.png (640, 480)
9003234.jpg (320, 239)
6000308.jpg (320, 240)
9000875.jpg (320, 240)
6000318.jpg (320, 214)
6000319.jpg (320, 240)
9004965.jpg (310, 320)
9002004.jpg (320, 214)
6000301.jpg (320, 240)
9001619.jpg (320, 240)
6000354.jpg (320, 240)
6000331.jpg (320, 240)
6000325.jpg (320, 240)
6000327.jpg (320, 240)
DifCanchaVSSS2.png (640, 480)
cocina.png (640, 480)
pisoMadera.png (640, 480)
9001317.jpg (320, 240)
6000341.jpg (320, 240)
9002577.jpg (320, 240)
6000320.jpg (320, 240)
9000029.jpg (240, 320)
6000314.jpg (320, 240)
8002274.jpg (240, 320)
9002972.jpg (320, 240)
9000989.j

#### Resize and normalize all bg images

In [33]:
bg_files = os.listdir(bg_folder)

size = (640, 480)
for bg in bg_files:
    bg_path = os.path.join(bg_folder, bg)
    with Image.open(bg_path) as img:
        resized_img = img.resize(size, resample = Image.BICUBIC)
        resized_img.save(bg_path)

#### Resize and normalize all fg_imgs

In [10]:
fg_size = (150, 150)
# Iterar sobre cada carpeta de primer plano
for folder_name in os.listdir(datasetPath):
    folder_path = os.path.join(datasetPath, folder_name)
    
    # Verificar si es una carpeta
    if os.path.isdir(folder_path):
        print(f"Procesando carpeta: {folder_name}")
        
        # Iterar sobre cada archivo en la carpeta
        for file_name in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file_name)
            
            # Verificar si es una imagen (por extensión)
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                print(f"  Procesando imagen: {file_name}")
                
                # Abrir la imagen con PIL
                try:
                    with Image.open(file_path) as fg:
                        resized_img = fg.resize(fg_size, resample = Image.BICUBIC)
                        resized_img.save(file_path)
                        print(f"    Tamaño: {img.size}, Modo: {img.mode}")
                except Exception as e:
                    print(f"    Error al abrir la imagen: {e}")

Procesando carpeta: pattern8
  Procesando imagen: WhatsApp Image 2025-04-14 at 8.42.28 PM (1) - Edited.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: pattern8.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: WhatsApp Image 2025-04-14 at 8.42.28 PM (2) - Edited.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: 533bfb22-915b-43b2-a651-6489a150e3d5 - Edited.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: d3e32343-eeb4-4d97-a7fa-c2e8bf840d03 - Edited.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: WhatsApp Image 2025-04-14 at 8.42.28 PM (4) - Edited.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: 8aa85c9d-3263-4ff0-be55-278993a47bd3 - Edited.png
    Error al abrir la imagen: name 'img' is not defined
  Procesando imagen: WhatsApp Image 2025-04-14 at 8.42.27 PM (1) - Edited - Edited.png
    Error a

In [11]:
#will mark error if already exist don't panic
if not os.path.exists(output_folder):
    os.makedirs(output_folder, exist_ok=True)
trainfolder = output_folder + "/train/"
testfolder = output_folder + "/test/"
validfolder = output_folder + "/valid/"
os.mkdir(trainfolder)
os.mkdir(testfolder)
os.mkdir(validfolder)
os.mkdir(trainfolder + "images/")
os.mkdir(trainfolder + "labels/")
os.mkdir(testfolder + "images/")
os.mkdir(testfolder + "labels/")
os.mkdir(validfolder + "images/")
os.mkdir(validfolder + "labels/")

FileExistsError: [Errno 17] File exists: '/home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/finalOutput//train/'

## Segmentation (NA)

In [ ]:
#Primera version

# Convert the image to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Apply thresholding to create a binary image
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY)

# Find contours in the binary image
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Iterate over the contours and draw them on the original image
for contour in contours:
    cv2.drawContours(image, [contour], -1, (0, 255, 0), 2)

# Display the image with contours
cv2.imshow("Segmentation", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Auto Label

In [22]:
images=[]
annotations=[]
annotations2=[]
annot_csv=[]

img_id=int(0)
anno_id=int(0)

rescaling_min = 0.10 #modify           #VERIFICAR TAMAÑO
rescaling_max = 0.23 #modify

# Ratios at which these values will be modified
brightness_ratio = 0.05
saturation_ratio = 0.05
hue_ratio = 0.02

#returns rotated list of pts
def rotated_point(pts_array, angle):
    #rotation matrix
    rot_mat = [[math.cos(angle), -math.sin(angle)], 
               [math.sin(angle), math.cos(angle)]]
    #Matrix1 rows x Matrix2 columns
    result = np.dot(rot_mat, pts_array)
    
    #return rotated list of pts
    return result


for j in range(100): #CANTIDAD DE IMGS A GENERAR 
    #create empty label file
    with open(f'{trainfolder}labels/{img_id}.txt', 'w') as file:
        pass
    #select hramdomly how many objects will be in an image              #MANTENERLO SIEMPRE EN 6 O REQUIERE VARIACIÓN???
    num_objects = random.randint(4, 6)
    #print("number of objects",num_objects)
    # Select random foreground images from the three folders, with replacement
    fg_categories = random.choices(objects_list, k=num_objects)
    
    fg_files_selected = []
    for category in fg_categories:
        fg_files_selected.append([category,random.choice(fg_files[category])])
    #print("seleccion",fg_files_selected)
    # Load the selected foreground images using Pillow
    fg_imgs = []
    for img in fg_files_selected:
        folder = [f[0] for f in fg_folders if f[1] == img[0]][0]
        fg_imgs.append([img[0],Image.open(folder + img[1]),folder+img[1]])

    for img in fg_imgs:
        fg_img=img[1]
        # Randomly resize and rotate the foreground images using Pillow's transform module
        angle = random.randint(-90,90) #rotation range
        scale = random.uniform(rescaling_min, rescaling_max)
        fg_img = fg_img.rotate(angle, resample=Image.BICUBIC, expand=True) #will influence obb
        fg_img = fg_img.resize((int(fg_img.width * scale), int(fg_img.height * scale)))
        fg_img = ImageEnhance.Brightness(fg_img).enhance(random.uniform(0.7, 1.3))
        fg_img = ImageEnhance.Contrast(fg_img).enhance(random.uniform(0.9, 1.1))
        fg_img = ImageEnhance.Color(fg_img).enhance(random.uniform(0.7, 1.3))
        fg_img = fg_img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.0, 0.5)))

        '''plt.imshow(fg_img) #DEBUG FOR FG IMG TO BE USED ----------------------
        fg_array = np.array(fg_img) 
        print(f"Canales de fg_img {fg_array.shape}")
        print(f"FG MODE {fg_img.mode}")
        plt.show() 
        #print(fg_array) #debug ----------------------------------------------------'''

        img[1] = fg_img

    # Load the background image using Pillow
    bg_files = os.listdir(bg_folder)
    bg_file = random.choice(bg_files)
    bg_img = Image.open(bg_folder + bg_file)
    

    # Define the maximum overlap as a percentage
    max_overlap_pct = 0 #in VSSS case you sest it to 0 so that NO overlap is allowed while generating the dataset, because it will never happen in the real competition

    # Define an array to keep track of occupied areas
    occupied = np.zeros((bg_img.height, bg_img.width))

    for img in fg_imgs:
        fg_img=img[1]
        # Calculate the maximum overlap area
        max_overlap_area = (fg_img.width * fg_img.height)

        seg_img = img[1]

        # Convert the image to a NumPy array
        img_arr = np.array(seg_img)
        # Create a binary mask of the non-transparent pixels
        mask = img_arr[:, :, 3] != 0

        # Convert the mask to a COCO format segmentation
        segmentation = []
        for i in range(mask.shape[0]): #iterates through the rows of the masc (height)
            for j in range(mask.shape[1]): #iterates through the columns of the masc (width)
                if mask[i, j]: #if the pixel belongs to the obj (fg_folder img)
                    segmentation.append(j) #adds the x coordinate
                    segmentation.append(i) #adds the y coordinate
        segmentation = [segmentation] #appends it into another array, for COCO format = [[x1, y1, x2, y2, x3, y3, ..., xn, yn]]

        # Calculate the area of the segmentation
        area = 0
        for i in range(len(segmentation[0]) // 2):
            x1 = segmentation[0][2 * i]
            y1 = segmentation[0][2 * i + 1]
            x2 = segmentation[0][(2 * i + 2) % len(segmentation[0])]
            y2 = segmentation[0][(2 * i + 3) % len(segmentation[0])]
            area += x1 * y2 - x2 * y1
        area = abs(area) / 2


        # Calculate the maximum allowed position for the top-left corner
        max_x = bg_img.width - fg_img.width
        max_y = bg_img.height - fg_img.height
        area = fg_img.width * fg_img.height #Area of the fg_img

        # Convert the image to a NumPy array and create a binary mask of the non-transparent pixels
        img_arr = np.array(fg_img)
        mask = img_arr[:, :, 3] > 0  # True where the pixel is visible (not transparent)

        fg_area = np.sum(mask)
        max_overlap_area = fg_area * (max_overlap_pct / 100)

        # Calculate the maximum allowed position for the top-left corner
        max_x = bg_img.width - fg_img.width
        max_y = bg_img.height - fg_img.height

        # Try placing the object with a limited number of attempts
        attempts = 0
        max_attempts = 50
        placed = False

        while attempts < max_attempts:
            x = random.randint(0, max_x)
            y = random.randint(0, max_y)

            # Check the overlap region
            overlap_region = occupied[y:y+fg_img.height, x:x+fg_img.width]
            overlap_area = np.sum(np.logical_and(overlap_region, mask))

            if overlap_area <= max_overlap_area:
                placed = True
                break

            attempts += 1

        if not placed:
            print(f"No se pudo colocar objeto sin solapamiento excesivo. Se omite este objeto.")
            continue

        # Marcar el área visible del objeto como ocupada
        occupied[y:y+fg_img.height, x:x+fg_img.width] = np.logical_or(occupied[y:y+fg_img.height, x:x+fg_img.width], mask)

        
        for i in range(0, len(segmentation[0])):
            if i % 2:
                segmentation[0][i]=int(segmentation[0][i]+y)
            else :
                segmentation[0][i]=int(segmentation[0][i]+x)
        # Update the occupied array
        occupied[y:y+fg_img.height, x:x+fg_img.width] = 1 

        #all imgs must be in the same img mode (in this case RGBA), if not done, transparency will NOT be correctly handled.
        if bg_img.mode in ('P', "RGB"):
            bg_img = bg_img.convert("RGBA")
      
        bg_img.paste(fg_img, (x, y), fg_img)
        
        #debug -----------------------------------
        bg_array = np.array(bg_img) 
        #print(f"Canales de bg_img {bg_array.shape}")
        #print(bg_array)
        #plt.imshow(bg_img) 
        #plt.show()
        #DEBUG --------------------------------------------

        # v8 data format--------------------- COORDINATES SHOULD BE NORMALIZED FOR YOLOV8
        bg_img.paste(fg_img, (x, y), fg_img)
        x_center_ann = (x+fg_img.width/2) / bg_img.width 
        y_center_ann = (y+fg_img.height/2) / bg_img.height
        width_ann = fg_img.width / bg_img.width
        height_ann = fg_img.height / bg_img.height 
        
        #Adaptation to YOLOv11 data format  (center is 0,0)-------------------------------
        #top-left corner                            (x1,y1)                  (x2, y2)
        x1 = x_center_ann - width_ann / 2              #o------------|-----------o
        y1 = y_center_ann - height_ann / 2             #|            |           |
        #top-right                                     #|            |           |
        x2 = x_center_ann + width_ann / 2              #|            |(0,0)      |
        y2 = y_center_ann - height_ann / 2             #|------------o-----------|
        #bottom right corner                           #|            |           |
        x3 = x_center_ann + width_ann / 2              #|            |           |
        y3 = y_center_ann + height_ann / 2             #|            |           |
        #bottom left corner                             o------------|-----------o
        x4 = x_center_ann - width_ann / 2           #(x4, y4)                     (x3, y3)
        y4 = y_center_ann + height_ann / 2

        #DEBUG
        draw = ImageDraw.Draw(bg_img)
        radius = 5  # Radio del punto
        '''draw.ellipse((x1 * bg_img.width - radius, y1 - radius, x1 + radius, y1 + radius), fill="blue")
        draw.ellipse((x2 - radius, y2 - radius, x2 + radius, y2 + radius), fill="blue")
        draw.ellipse((x3 - radius, y3 - radius, x3 + radius, y3 + radius), fill="blue")
        draw.ellipse((x4 - radius, y4 - radius, x4 + radius, y4 + radius), fill="blue")
        draw.rectangle([(x1, y1), (x3, y3)], outline="red", width=2)'''

        #Rotated points
        #angle is the variable going from -90° to 90°
        angle_rad = math.radians(angle) #because math works with radians and ur using cos, sin.
    
        notNormX = x_center_ann * bg_img.width
        notNormY = y_center_ann * bg_img.height
        radius = 5  # Radio del círculo
        #draw.ellipse((notNormX - radius, notNormY - radius, notNormX + radius, notNormY + radius), fill="blue")
        #print(f"CENTRO SII: {x_center_ann, y_center_ann}")
        #print(f"CENTRO SIN NORMALIZAR: {notNormX}, {notNormY}")
        
        
        #save rotated pts
        with open(f'{trainfolder}labels/{img_id}.txt', 'a') as f: #CHECK ANNOTATIONS FORMAT FOR V11----------------start from here, also the dictionary
            f.write(f"{annotations_ID[img[0]]} {x_center_ann} {y_center_ann} {width_ann} {height_ann}\n") 
        annotations2.append({"id": anno_id,"image_id": img_id,"category_id": annotations_ID[img[0]],"bbox": [x, y, fg_img.width, fg_img.height],"segmentation": segmentation,"area": area,"iscrowd": 0})
        annotations.append({"id": anno_id,"image_id": img_id,"category_id": annotations_ID[img[0]],"bbox": [x, y, fg_img.width, fg_img.height],"segmentation": [],"area": fg_img.height*fg_img.width,"iscrowd": 0})
        annot_csv.append(["TRAIN", output_folder + str(img_id)+".jpg", img[0], x/bg_img.width, y/bg_img.height,"","",(x+fg_img.width)/bg_img.width, (y+fg_img.height)/bg_img.height])
        anno_id=anno_id+1

    #print(f"BG_MODE {bg_img.mode}") #DEBUG ------------------------------------------------------------------------
    if bg_img.mode in ("P", "RGBA"): #Verify img mode to see if it has the alpha channel
        bg_img = bg_img.convert("RGB") #Convert image to RGB for jpg format
    bg_img.save(f"{trainfolder}images/"+str(img_id)+".jpg", quality=100)
    images.append({"id": img_id, "file_name": str(img_id)+".jpg","height": bg_img.height,"width": bg_img.width})
    img_id=img_id+1
    #print(img_id)

#making data.yaml
data = dict(  #puedes usar ordered dict para forzar cierto orden cuando lo metas el yaml
    train = f"{trainfolder}images",
    val = f"{validfolder}images",
    test = f"{validfolder}images",
    nc = len(annotations_ID),
    names = list(annotations_ID.keys())
    )
#storing
with open(f'{output_folder}data.yaml', 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)

SplitTrainValidation

In [40]:
import os
import shutil #file and directory operations
import random

#represent how much of 1.0 (100%) of the train data will be used in validation and test
'''data asignation may vary depending on dataset length, 
for shorter datasets a higher percentage is recommended, 
while with bigger datasets a lower percentage is good
The percentage split for validation and test datasets depends on the size of your dataset and the specific use case. Here are some general recommendations:

Common Splits:
1. **Small Datasets** (less than 1,000 samples):
   - **Training**: 70-80%
   - **Validation**: 10-15%
   - **Test**: 10-15%

2. **Medium Datasets** (1,000 to 10,000 samples):
   - **Training**: 70-80%
   - **Validation**: 10-20%
   - **Test**: 10-20%

3. **Large Datasets** (more than 10,000 samples):
   - **Training**: 80-90%
   - **Validation**: 5-10%
   - **Test**: 5-10%
'''
validation = 0.1 #using 10% of the dataset 
test = 0.1 #using 10% of the data

# Assumes test has 100% of data. WHY??-------------------------------------------------------------
output_folder = "/home/daniela/Desktop/VSSS/larc-vsss-2025/dataset-generator/Resources/finalOutput/"
#train folder and it's respective imgs and labels inside folders
trainfolder = output_folder + "train/"
trainfolderimgs = trainfolder + "images/"
trainfolderlabels = trainfolder + "labels/"
#test folder and it's respective imgs and labels inside folders
testfolder = output_folder + "test/"
testfolderimgs = testfolder + "images/"
testfolderlabels = testfolder + "labels/"
#valid folder and it's respective imgs and labels inside folders
validfolder = output_folder + "valid/"
validfolderimgs = validfolder + "images/"
validfolderlabels = validfolder + "labels/"

fullSize = len(os.listdir(trainfolderimgs))
validSize = int(fullSize * validation)
testSize = int(fullSize * test)

for i in range(validSize):
    filelist = os.listdir(trainfolderimgs)
    #randomize file list, to not pick files in order
    random.shuffle(filelist)
    filetomove = filelist[i]
    #take out .jpg, .png, etc
    filetomovename = filetomove[:-4]
    #move images
    shutil.move(f"{trainfolderimgs}{filetomove}", f"{validfolderimgs}{filetomove}")
    #move labels
    shutil.move(f"{trainfolderlabels}{filetomovename}.txt", f"{validfolderlabels}{filetomovename}.txt")
for i in range(testSize):
    filetomove = os.listdir(trainfolderimgs)[i]
    #take out .jpg, .png, etc
    filetomovename = filetomove[:-4]
    #move images
    shutil.move(f"{trainfolderimgs}{filetomove}", f"{testfolderimgs}{filetomove}")
    #move labels
    shutil.move(f"{trainfolderlabels}{filetomovename}.txt", f"{testfolderlabels}{filetomovename}.txt")

#Validation
print(f"Train size is now: {len(os.listdir(trainfolderimgs))}")
print(f"Validation size is now: {len(os.listdir(validfolderimgs))}")
print(f"Test size is now: {len(os.listdir(testfolderimgs))}")

Train size is now: 4000
Validation size is now: 864
Test size is now: 719


In [38]:
import sys
print(sys.executable)

/bin/python3
